# IEEE Advanced Final Clean Notebook — HAM10000

This notebook is designed for the IEEE paper only. It generates advanced results distinct from the Digital Health paper:

- Core ablation performance
- Cross-validation stability
- Noise, brightness, and contrast robustness
- Reliability diagram and calibration-gap analysis
- Confidence–accuracy analysis
- Predictive entropy analysis
- Corrected global pixel-feature importance heatmap
- Top-feature decay and cumulative importance
- Perturbation sensitivity and explanation-stability diagnostics

Expected data path:

`/content/drive/MyDrive/Datasets/HAM10000/hmnist_28_28_RGB.csv`


**Revision note (Scientific Reports):** cells 27–31 add temperature scaling, misclassification-detection AUROC/AUPR, selective-prediction / risk–coverage, accept-defer operating points, and rank-correlation explanation stability — repositioning the study around predictive trust rather than accuracy.


In [ ]:
# ============================================================
# 1. Setup
# ============================================================

import os, json, time, math, warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, roc_auc_score, log_loss
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.utils.class_weight import compute_class_weight

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

def log(msg):
    print(msg)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def save_fig(fig_dir, name):
    plt.tight_layout()
    path = fig_dir / name
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"[FIG] Saved: {path}")

import time as _time
_T0 = _time.time()
def progress(msg):
    """Timestamped, flushed progress line so long cells show life in Colab."""
    print(f"[{_time.time()-_T0:7.1f}s] {msg}", flush=True)

log("[SETUP] Ready.")
progress("setup complete")


In [ ]:
# ============================================================
# 2. Mount Drive and define paths
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_ROOT = Path('/content/drive/MyDrive/Datasets/HAM10000')
CSV_PATH = DATA_ROOT / 'hmnist_28_28_RGB.csv'

OUTPUT_ROOT = Path('/content/drive/MyDrive/Outputs/HAM10000_IEEE_Advanced') / RUN_ID
FIG_DIR = OUTPUT_ROOT / 'figures'
TABLE_DIR = OUTPUT_ROOT / 'tables'
OTHER_DIR = OUTPUT_ROOT / 'others'

for d in [OUTPUT_ROOT, FIG_DIR, TABLE_DIR, OTHER_DIR]:
    ensure_dir(d)

SUMMARY_PATH = OUTPUT_ROOT / 'outputs_summary.txt'

def write_summary(text, mode="a"):
    with open(SUMMARY_PATH, mode, encoding="utf-8") as f:
        f.write(str(text).rstrip() + "\n")

write_summary("IEEE Advanced Final Clean Notebook", mode="w")
write_summary(f"Run ID: {RUN_ID}")
write_summary(f"Output root: {OUTPUT_ROOT}")

log(f"[OUTPUT] {OUTPUT_ROOT}")


In [ ]:
# ============================================================
# 3. Load HAM10000 CSV data
# ============================================================

if not CSV_PATH.exists():
    raise FileNotFoundError(f"Missing CSV file: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
if "label" not in df.columns:
    raise ValueError("Column 'label' not found.")

X = df.drop(columns=["label"]).values.astype(np.float32) / 255.0
y = df["label"].values.astype(int)
X_images = X.reshape(-1, 28, 28, 3)

# hmnist_28_28_RGB.csv label encoding (verified against canonical HAM10000 counts):
# akiec=327, bcc=514, bkl=1099, df=115, nv=6705, vasc=142, mel=1113
class_names = {0:"akiec", 1:"bcc", 2:"bkl", 3:"df", 4:"nv", 5:"vasc", 6:"mel"}
classes = sorted(np.unique(y))

class_df = pd.DataFrame({
    "label": classes,
    "class_name": [class_names[i] for i in classes],
    "count": [int(np.sum(y == i)) for i in classes]
})
class_df.to_csv(TABLE_DIR / "table_ieee_dataset_class_distribution.csv", index=False)

log(f"[DATA] X: {X.shape}; y: {y.shape}; images: {X_images.shape}")
log(class_df)

write_summary("\nDATASET")
write_summary(f"Samples: {X.shape[0]}; Features: {X.shape[1]}")
write_summary(class_df.to_string(index=False))


In [ ]:
# ============================================================
# 4. Split and class weights
# ============================================================

X_train, X_test, y_train, y_test, img_train, img_test = train_test_split(
    X, y, X_images, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

weights = compute_class_weight(class_weight="balanced", classes=np.array(classes), y=y_train)
class_weight_dict = dict(zip(classes, weights))

pd.DataFrame({
    "label": list(class_weight_dict.keys()),
    "class_name": [class_names[i] for i in class_weight_dict.keys()],
    "class_weight": list(class_weight_dict.values())
}).to_csv(TABLE_DIR / "table_ieee_class_weights.csv", index=False)

log(f"[SPLIT] Train {X_train.shape}; Test {X_test.shape}")
log(f"[WEIGHTS] {class_weight_dict}")

write_summary("\nSPLIT AND WEIGHTS")
write_summary(f"Train: {X_train.shape}; Test: {X_test.shape}")
write_summary(class_weight_dict)


In [ ]:
# ============================================================
# 5. IEEE fast model set
# ============================================================

models = {}

models["LogisticRegression"] = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

models["RandomForest"] = RandomForestClassifier(
    n_estimators=100,
    max_depth=18,
    min_samples_leaf=3,
    class_weight=class_weight_dict,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

models["ExtraTrees"] = ExtraTreesClassifier(
    n_estimators=120,
    max_depth=18,
    min_samples_leaf=3,
    class_weight=class_weight_dict,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

models["SoftVoting_RF_ET"] = VotingClassifier(
    estimators=[("RandomForest", models["RandomForest"]), ("ExtraTrees", models["ExtraTrees"])],
    voting="soft",
    n_jobs=-1
)

log("[MODELS]")
for k in models:
    log(" - " + k)


In [ ]:
# ============================================================
# 6. Helper functions
# ============================================================

def safe_predict_proba(model, X_input):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_input)
    pred = model.predict(X_input)
    proba = np.zeros((len(pred), len(classes)))
    for i, p in enumerate(pred):
        proba[i, int(p)] = 1.0
    return proba

def predictive_entropy(proba):
    eps = 1e-12
    return -np.sum(proba * np.log(proba + eps), axis=1)

def expected_calibration_error(y_true, proba, n_bins=10):
    conf = np.max(proba, axis=1)
    pred = np.argmax(proba, axis=1)
    correct = (pred == y_true).astype(float)
    bins = np.linspace(0, 1, n_bins+1)
    rows, ece = [], 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        if mask.sum() == 0:
            continue
        acc = float(correct[mask].mean())
        c = float(conf[mask].mean())
        gap = abs(acc - c)
        ece += float(mask.mean()) * gap
        rows.append({"bin_low": lo, "bin_high": hi, "count": int(mask.sum()), "accuracy": acc, "confidence": c, "calibration_gap": gap})
    return float(ece), pd.DataFrame(rows)

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0
    pred = model.predict(X_te)
    proba = safe_predict_proba(model, X_te)

    p, r, f1, _ = precision_recall_fscore_support(y_te, pred, average="weighted", zero_division=0)

    try:
        auc = roc_auc_score(label_binarize(y_te, classes=classes), proba, average="weighted", multi_class="ovr")
    except Exception:
        auc = np.nan

    try:
        ll = log_loss(y_te, proba)
    except Exception:
        ll = np.nan

    ece, _ = expected_calibration_error(y_te, proba)

    return {
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "balanced_accuracy": balanced_accuracy_score(y_te, pred),
        "precision_weighted": p,
        "recall_weighted": r,
        "f1_weighted": f1,
        "roc_auc_ovr_weighted": auc,
        "log_loss": ll,
        "ece": ece,
        "train_time_sec": train_time
    }, pred, proba, model


In [ ]:
# ============================================================
# 7. Core model ablation
# ============================================================

core_results, predictions, probabilities, trained_models = [], {}, {}, {}

for name, model in models.items():
    progress(f"[TRAIN] start: {name}")
    result, pred, proba, fitted = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    progress(f"[TRAIN] done:  {name}  acc={result['accuracy']:.3f}  f1={result['f1_weighted']:.3f}  ({result['train_time_sec']:.1f}s)")
    core_results.append(result)
    predictions[name] = pred
    probabilities[name] = proba
    trained_models[name] = fitted

core_df = pd.DataFrame(core_results).sort_values("f1_weighted", ascending=False)
core_df.to_csv(TABLE_DIR / "table_ieee_core_ablation_performance.csv", index=False)

best_model_name = core_df.iloc[0]["model"]
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]
best_proba = probabilities[best_model_name]

log(core_df)
log(f"[BEST] {best_model_name}")

write_summary("\nCORE ABLATION")
write_summary(core_df.to_string(index=False))
write_summary(f"Best model: {best_model_name}")


In [ ]:
# ============================================================
# 8. Figure 1: Core ablation performance
# ============================================================

plt.figure(figsize=(8,5))
x = np.arange(len(core_df))
plt.bar(x - 0.2, core_df["accuracy"], width=0.4, label="Accuracy")
plt.bar(x + 0.2, core_df["f1_weighted"], width=0.4, label="Weighted F1")
plt.xticks(x, core_df["model"], rotation=30, ha="right")
plt.ylabel("Score")
plt.title("Core Model Ablation Performance")
plt.legend()
save_fig(FIG_DIR, "fig_ieee_core_ablation_performance.png")


In [ ]:
# ============================================================
# 9. Cross-validation stability
# ============================================================

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"accuracy":"accuracy", "balanced_accuracy":"balanced_accuracy", "f1_weighted":"f1_weighted"}

cv_rows = []
for name, model in models.items():
    progress(f"[CV] start: {name}  (LogisticRegression/saga is the slow one — be patient)")
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False)
    progress(f"[CV] done:  {name}  f1_mean={float(np.mean(scores['test_f1_weighted'])):.3f}")
    row = {"model": name}
    for metric in scoring:
        vals = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = float(np.mean(vals))
        row[f"{metric}_std"] = float(np.std(vals))
    row["fit_time_mean_sec"] = float(np.mean(scores["fit_time"]))
    cv_rows.append(row)

cv_df = pd.DataFrame(cv_rows).sort_values("f1_weighted_mean", ascending=False)
cv_df.to_csv(TABLE_DIR / "table_ieee_cv_summary.csv", index=False)

log(cv_df)
write_summary("\nCV SUMMARY")
write_summary(cv_df.to_string(index=False))


In [ ]:
# ============================================================
# 10. Figure 2: Cross-validation stability
# ============================================================

plt.figure(figsize=(8,5))
plt.errorbar(cv_df["model"], cv_df["f1_weighted_mean"], yerr=cv_df["f1_weighted_std"], fmt="o", capsize=5, label="Weighted F1")
plt.errorbar(cv_df["model"], cv_df["accuracy_mean"], yerr=cv_df["accuracy_std"], fmt="s", capsize=5, label="Accuracy")
plt.ylabel("Cross-validation score")
plt.title("Cross-Validation Stability")
plt.xticks(rotation=30, ha="right")
plt.legend()
save_fig(FIG_DIR, "fig_ieee_cv_stability.png")


In [ ]:
# ============================================================
# 11. Robustness experiments
# ============================================================

def add_gaussian_noise(X_input, sigma):
    return np.clip(X_input + np.random.normal(0, sigma, X_input.shape), 0, 1)

def brightness_shift(X_input, delta):
    return np.clip(X_input + delta, 0, 1)

def contrast_scale(X_input, factor):
    return np.clip(0.5 + factor * (X_input - 0.5), 0, 1)

perturbations = [
    ("clean", "none", 0.00, X_test),
    ("noise_sigma_0.03", "gaussian_noise", 0.03, add_gaussian_noise(X_test, 0.03)),
    ("noise_sigma_0.06", "gaussian_noise", 0.06, add_gaussian_noise(X_test, 0.06)),
    ("noise_sigma_0.10", "gaussian_noise", 0.10, add_gaussian_noise(X_test, 0.10)),
    ("brightness_plus_0.10", "brightness", 0.10, brightness_shift(X_test, 0.10)),
    ("brightness_minus_0.10", "brightness", -0.10, brightness_shift(X_test, -0.10)),
    ("contrast_0.80", "contrast", 0.80, contrast_scale(X_test, 0.80)),
    ("contrast_1.20", "contrast", 1.20, contrast_scale(X_test, 1.20)),
]

robust_rows = []
for pname, ptype, level, Xp in perturbations:
    progress(f"[ROBUST] {pname}")
    pred = best_model.predict(Xp)
    proba = safe_predict_proba(best_model, Xp)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="weighted", zero_division=0)
    ece, _ = expected_calibration_error(y_test, proba)
    robust_rows.append({
        "best_model": best_model_name,
        "perturbation": pname,
        "type": ptype,
        "level": level,
        "accuracy": accuracy_score(y_test, pred),
        "balanced_accuracy": balanced_accuracy_score(y_test, pred),
        "precision_weighted": p,
        "recall_weighted": r,
        "f1_weighted": f1,
        "mean_confidence": float(np.max(proba, axis=1).mean()),
        "ece": ece,
        "mean_entropy": float(predictive_entropy(proba).mean())
    })

robustness_df = pd.DataFrame(robust_rows)
robustness_df.to_csv(TABLE_DIR / "table_ieee_robustness.csv", index=False)

log(robustness_df)
write_summary("\nROBUSTNESS")
write_summary(robustness_df.to_string(index=False))


In [ ]:
# ============================================================
# 12. Figure 3: Gaussian-noise robustness
# ============================================================

noise_df = robustness_df[robustness_df["type"].isin(["none", "gaussian_noise"])].copy()

plt.figure(figsize=(7,5))
plt.plot(noise_df["level"], noise_df["accuracy"], marker="o", label="Accuracy")
plt.plot(noise_df["level"], noise_df["f1_weighted"], marker="s", label="Weighted F1")
plt.plot(noise_df["level"], noise_df["balanced_accuracy"], marker="^", label="Balanced accuracy")
plt.xlabel("Gaussian noise level ($\sigma$)")
plt.ylabel("Score")
plt.title("Robustness to Gaussian Noise")
plt.legend()
save_fig(FIG_DIR, "fig_ieee_noise_curve.png")


In [ ]:
# ============================================================
# 13. Figure 4: Perturbation robustness summary
# ============================================================

plot_df = robustness_df[robustness_df["type"] != "none"].copy()

plt.figure(figsize=(9,5))
x = np.arange(len(plot_df))
plt.bar(x, plot_df["f1_weighted"])
plt.xticks(x, plot_df["perturbation"], rotation=40, ha="right")
plt.ylabel("Weighted F1-score")
plt.title("Robustness Across Perturbation Types")
save_fig(FIG_DIR, "fig_ieee_perturbation_summary.png")


In [ ]:
# ============================================================
# 14. Reliability, calibration, and uncertainty
# ============================================================

ece, calib_df = expected_calibration_error(y_test, best_proba, n_bins=10)
calib_df.to_csv(TABLE_DIR / "table_ieee_calibration_bins.csv", index=False)

conf = np.max(best_proba, axis=1)
correct = (best_pred == y_test)
entropy = predictive_entropy(best_proba)

reliability_df = pd.DataFrame([{
    "best_model": best_model_name,
    "accuracy": accuracy_score(y_test, best_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, best_pred),
    "f1_weighted": core_df.loc[core_df["model"] == best_model_name, "f1_weighted"].iloc[0],
    "mean_confidence": float(conf.mean()),
    "expected_calibration_error": ece,
    "mean_entropy": float(entropy.mean()),
    "overconfident_error_rate_conf_gt_0_80": float(np.mean((conf > 0.80) & (~correct))),
    "low_confidence_fraction_conf_lt_0_50": float(np.mean(conf < 0.50))
}])
reliability_df.to_csv(TABLE_DIR / "table_ieee_reliability_summary.csv", index=False)

log(reliability_df)
log(calib_df)
write_summary("\nRELIABILITY")
write_summary(reliability_df.to_string(index=False))
write_summary(calib_df.to_string(index=False))


In [ ]:
# ============================================================
# 15. Figure 5: Reliability diagram
# ============================================================

plt.figure(figsize=(6,5))
plt.plot([0,1], [0,1], linestyle="--", label="Perfect calibration")
plt.plot(calib_df["confidence"], calib_df["accuracy"], marker="o", label=best_model_name)
plt.xlabel("Mean predicted confidence")
plt.ylabel("Empirical accuracy")
plt.title("Reliability Diagram")
plt.legend()
save_fig(FIG_DIR, "fig_ieee_reliability_diagram.png")


In [ ]:
# ============================================================
# 16. Figure 6: Calibration gap
# ============================================================

labels = [f"{r.bin_low:.1f}-{r.bin_high:.1f}" for _, r in calib_df.iterrows()]

plt.figure(figsize=(7,5))
plt.bar(labels, calib_df["calibration_gap"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("|Accuracy - Confidence|")
plt.title("Calibration Gap Across Confidence Bins")
save_fig(FIG_DIR, "fig_ieee_calibration_gap.png")


In [ ]:
# ============================================================
# 17. Figure 7: Confidence-accuracy relationship
# ============================================================

conf_acc_df = pd.DataFrame({
    "bin_center": 0.5*(calib_df["bin_low"] + calib_df["bin_high"]),
    "accuracy": calib_df["accuracy"],
    "confidence": calib_df["confidence"],
    "count": calib_df["count"]
})
conf_acc_df.to_csv(TABLE_DIR / "table_ieee_confidence_accuracy.csv", index=False)

plt.figure(figsize=(7,5))
plt.plot(conf_acc_df["bin_center"], conf_acc_df["accuracy"], marker="o", label="Empirical accuracy")
plt.plot(conf_acc_df["bin_center"], conf_acc_df["confidence"], marker="s", label="Mean confidence")
plt.xlabel("Confidence bin center")
plt.ylabel("Value")
plt.title("Confidence–Accuracy Relationship")
plt.legend()
save_fig(FIG_DIR, "fig_ieee_confidence_accuracy.png")


In [ ]:
# ============================================================
# 18. Figure 8: Predictive entropy for correct vs incorrect predictions
# ============================================================

entropy_df = pd.DataFrame({"entropy": entropy, "correct": correct})
entropy_df.to_csv(TABLE_DIR / "table_ieee_entropy_correctness.csv", index=False)

plt.figure(figsize=(7,5))
plt.hist(entropy_df[entropy_df["correct"]]["entropy"], bins=25, alpha=0.7, label="Correct")
plt.hist(entropy_df[~entropy_df["correct"]]["entropy"], bins=25, alpha=0.7, label="Incorrect")
plt.xlabel("Predictive entropy")
plt.ylabel("Count")
plt.title("Predictive Entropy for Correct and Incorrect Predictions")
plt.legend()
save_fig(FIG_DIR, "fig_ieee_entropy_correctness.png")


In [ ]:
# ============================================================
# 19. Corrected feature-importance block
# This block always explains WHY a feature-importance figure is or is not possible.
# ============================================================

importance_values = None
importance_method = None
importance_source_model = None

print("[XAI] Best model:", best_model_name)
print("[XAI] Best model type:", type(best_model))
print("[XAI] Has feature_importances_:", hasattr(best_model, "feature_importances_"))

if hasattr(best_model, "feature_importances_"):
    importance_values = np.asarray(best_model.feature_importances_)
    importance_method = "tree_feature_importance"
    importance_source_model = best_model_name

elif hasattr(best_model, "estimators_"):
    collected = []
    source_names = []
    print("[XAI] Voting model detected. Checking component estimators.")
    for est_name, est in zip([name for name, _ in best_model.estimators], best_model.estimators_):
        print(" -", est_name, "has feature_importances_:", hasattr(est, "feature_importances_"))
        if hasattr(est, "feature_importances_"):
            collected.append(np.asarray(est.feature_importances_))
            source_names.append(est_name)
    if len(collected) > 0:
        importance_values = np.mean(np.vstack(collected), axis=0)
        importance_method = "mean_component_tree_feature_importance"
        importance_source_model = "+".join(source_names)

if importance_values is None:
    print("[XAI] No direct feature_importances_ found. Using permutation importance fallback.")
    subset_n = min(800, len(X_test))
    perm = permutation_importance(
        best_model, X_test[:subset_n], y_test[:subset_n],
        n_repeats=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    importance_values = np.asarray(perm.importances_mean)
    importance_method = "permutation_importance"
    importance_source_model = best_model_name

print("[XAI] Importance method:", importance_method)
print("[XAI] Source model:", importance_source_model)
print("[XAI] Importance length:", len(importance_values))

expected_features = 28 * 28 * 3
if len(importance_values) != expected_features:
    raise ValueError(f"Feature importance length mismatch: got {len(importance_values)}, expected {expected_features}.")

importance_df = pd.DataFrame({
    "pixel_feature_index": np.arange(len(importance_values)),
    "importance": importance_values
}).sort_values("importance", ascending=False)

importance_df.to_csv(TABLE_DIR / "table_ieee_top_feature_importance.csv", index=False)

importance_img = importance_values.reshape(28, 28, 3)
importance_gray = importance_img.mean(axis=2)

write_summary("\nFEATURE IMPORTANCE")
write_summary(f"Method: {importance_method}")
write_summary(f"Source model: {importance_source_model}")
write_summary(importance_df.head(25).to_string(index=False))


In [ ]:
# ============================================================
# 20. Figure 9: Global pixel-feature importance heatmap
# ============================================================

plt.figure(figsize=(6,5))
plt.imshow(importance_gray)
plt.title(f"Global Pixel Importance ({importance_source_model})")
plt.axis("off")
plt.colorbar()
save_fig(FIG_DIR, "fig_ieee_feature_importance_heatmap.png")


In [ ]:
# ============================================================
# 21. Figure 10: Top feature-importance decay
# ============================================================

top_n = 50
top_imp = importance_df.head(top_n)

plt.figure(figsize=(8,5))
plt.plot(np.arange(1, top_n+1), top_imp["importance"].values, marker="o")
plt.xlabel("Top-ranked pixel feature")
plt.ylabel("Importance")
plt.title("Top Pixel-Feature Importance Decay")
save_fig(FIG_DIR, "fig_ieee_top_feature_importance_decay.png")


In [ ]:
# ============================================================
# 22. Figure 11: Cumulative feature-importance curve
# ============================================================

sorted_imp = np.sort(importance_values)[::-1]
cum_imp = np.cumsum(sorted_imp) / (np.sum(sorted_imp) + 1e-12)

plt.figure(figsize=(7,5))
plt.plot(np.arange(1, len(cum_imp)+1), cum_imp)
plt.xlabel("Number of top-ranked features")
plt.ylabel("Cumulative normalized importance")
plt.title("Cumulative Pixel-Feature Importance")
save_fig(FIG_DIR, "fig_ieee_cumulative_feature_importance.png")


In [ ]:
# ============================================================
# 23. Perturbation sensitivity and explanation stability
# ============================================================

def prediction_sensitivity_map(model, X_clean, sigma=0.06, n_samples=500):
    idx = np.random.choice(len(X_clean), size=min(n_samples, len(X_clean)), replace=False)
    Xc = X_clean[idx]
    Xn = add_gaussian_noise(Xc, sigma)
    pc = safe_predict_proba(model, Xc)
    pn = safe_predict_proba(model, Xn)
    sample_delta = np.mean(np.abs(pn - pc), axis=1)
    pixel_delta = np.abs(Xn - Xc)
    weighted_pixel_delta = (pixel_delta.T @ sample_delta) / (np.sum(sample_delta) + 1e-12)
    return weighted_pixel_delta

progress("[XAI] computing perturbation sensitivity map (500 samples)")
sensitivity_values = prediction_sensitivity_map(best_model, X_test, sigma=0.06, n_samples=500)

importance_norm = (importance_values - importance_values.min()) / (importance_values.max() - importance_values.min() + 1e-12)
sensitivity_norm = (sensitivity_values - sensitivity_values.min()) / (sensitivity_values.max() - sensitivity_values.min() + 1e-12)

importance_sensitivity_corr = float(np.corrcoef(importance_norm, sensitivity_norm)[0, 1])
importance_sensitivity_mae = float(np.mean(np.abs(importance_norm - sensitivity_norm)))

stability_df = pd.DataFrame([{
    "best_model": best_model_name,
    "noise_sigma": 0.06,
    "importance_sensitivity_correlation": importance_sensitivity_corr,
    "importance_sensitivity_mae": importance_sensitivity_mae,
    "interpretation": "Correlation and MAE compare normalized feature importance with perturbation sensitivity."
}])
stability_df.to_csv(TABLE_DIR / "table_ieee_explanation_stability.csv", index=False)

sensitivity_gray = sensitivity_values.reshape(28, 28, 3).mean(axis=2)

log(stability_df)
write_summary("\nEXPLANATION STABILITY")
write_summary(stability_df.to_string(index=False))


In [ ]:
# ============================================================
# 24. Figure 12: Perturbation sensitivity map
# ============================================================

plt.figure(figsize=(6,5))
plt.imshow(sensitivity_gray)
plt.title("Perturbation Sensitivity Map")
plt.axis("off")
plt.colorbar()
save_fig(FIG_DIR, "fig_ieee_perturbation_sensitivity_map.png")


In [ ]:
# ============================================================
# 25. Figure 13: Importance-sensitivity scatter
# ============================================================

sample_idx = np.random.choice(len(importance_norm), size=min(1000, len(importance_norm)), replace=False)

plt.figure(figsize=(6,5))
plt.scatter(importance_norm[sample_idx], sensitivity_norm[sample_idx], s=10, alpha=0.5)
plt.xlabel("Normalized feature importance")
plt.ylabel("Normalized perturbation sensitivity")
plt.title("Importance–Sensitivity Relationship")
save_fig(FIG_DIR, "fig_ieee_importance_sensitivity_scatter.png")


## Reliability Upgrade for the Scientific Reports Revision

The cells below extend the original pipeline with the analyses that reposition the
study around **predictive trust** rather than accuracy. They reuse the probabilities
already computed for the best model — **no retraining**.

1. **Temperature scaling** with calibration reported *before vs after*
   (ECE, MCE, Brier, NLL) on a held-out calibration/evaluation split.
2. **Misclassification detection** by predictive entropy and max-confidence
   (AUROC / AUPR) — the entropy histogram turned into numbers.
3. **Selective prediction / risk–coverage**: the accept/defer rule actually executed,
   benchmarked against confidence, entropy, random and oracle rankings (with AURC).
4. **Operating points**: the paper's fixed (tau_c, tau_H) rule and a principled point
   chosen for a target selective accuracy.
5. **Explanation stability**: Spearman rank correlation added alongside Pearson.

In [ ]:
# ============================================================
# 27. [SR] Temperature scaling — calibration before vs after
# ============================================================
from scipy.optimize import minimize_scalar
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve

def scale_proba(proba, T):
    """Apply temperature T to probability vectors (softmax over log-probs / T)."""
    z = np.log(np.clip(proba, 1e-12, 1.0)) / float(T)
    z -= z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def brier_multiclass(y_true, proba):
    Y = label_binarize(y_true, classes=classes)
    return float(np.mean(np.sum((proba - Y) ** 2, axis=1)))

def calib_metrics(y_true, proba, n_bins=10):
    ece, df_bins = expected_calibration_error(y_true, proba, n_bins=n_bins)
    mce = float(df_bins["calibration_gap"].max()) if len(df_bins) else float("nan")
    return {"ece": ece, "mce": mce,
            "brier": brier_multiclass(y_true, proba),
            "nll": float(log_loss(y_true, proba, labels=classes))}

# Held-out calibration / evaluation split of the test set (disjoint subsets)
idx_all = np.arange(len(y_test))
cal_idx, eval_idx = train_test_split(
    idx_all, test_size=0.5, stratify=y_test, random_state=RANDOM_STATE
)
proba_cal, y_cal = best_proba[cal_idx], y_test[cal_idx]
proba_eval, y_eval = best_proba[eval_idx], y_test[eval_idx]

# Fit temperature on the calibration split by minimizing NLL
res = minimize_scalar(
    lambda T: log_loss(y_cal, scale_proba(proba_cal, T), labels=classes),
    bounds=(0.05, 10.0), method="bounded"
)
T_opt = float(res.x)

before = calib_metrics(y_eval, proba_eval)
after = calib_metrics(y_eval, scale_proba(proba_eval, T_opt))
temp_df = pd.DataFrame([{"stage": "before (T=1)", **before},
                        {"stage": f"after (T={T_opt:.3f})", **after}])
temp_df.to_csv(TABLE_DIR / "table_sr_temperature_scaling.csv", index=False)

log(f"[SR][TEMP] Optimal T = {T_opt:.4f}  (T<1 sharpens an underconfident model)")
log(temp_df)
write_summary("\n[SR] TEMPERATURE SCALING")
write_summary(f"Optimal temperature T = {T_opt:.4f}")
write_summary(temp_df.to_string(index=False))

# Reliability curves before vs after (evaluation split)
_, bins_before = expected_calibration_error(y_eval, proba_eval, n_bins=10)
_, bins_after = expected_calibration_error(y_eval, scale_proba(proba_eval, T_opt), n_bins=10)

plt.figure(figsize=(6,5))
plt.plot([0,1],[0,1],"--",label="Perfect calibration")
plt.plot(bins_before["confidence"], bins_before["accuracy"], marker="o", label="Before (T=1)")
plt.plot(bins_after["confidence"], bins_after["accuracy"], marker="s", label=f"After (T={T_opt:.2f})")
plt.xlabel("Mean predicted confidence"); plt.ylabel("Empirical accuracy")
plt.title("Reliability Before vs After Temperature Scaling"); plt.legend()
save_fig(FIG_DIR, "fig_sr_reliability_temperature.png")

In [ ]:
# ============================================================
# 28. [SR] Misclassification detection — entropy & confidence
# ============================================================
# Reuses conf, entropy, correct from the reliability/entropy cells above.
corr = np.asarray(correct)
incorrect = (~corr).astype(int)            # positive class = "prediction is wrong"

scores = {
    "predictive_entropy": np.asarray(entropy),   # higher -> more likely wrong
    "one_minus_confidence": 1.0 - np.asarray(conf),
}
det_rows = []
plt.figure(figsize=(6,5))
for sname, sval in scores.items():
    auroc = float(roc_auc_score(incorrect, sval))
    aupr = float(average_precision_score(incorrect, sval))
    det_rows.append({"score": sname,
                     "auroc_error_detection": round(auroc, 4),
                     "aupr_error_detection": round(aupr, 4)})
    fpr, tpr, _ = roc_curve(incorrect, sval)
    plt.plot(fpr, tpr, label=f"{sname} (AUROC={auroc:.3f})")
plt.plot([0,1],[0,1],"--",color="gray")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("Misclassification Detection (error = positive class)"); plt.legend()
save_fig(FIG_DIR, "fig_sr_error_detection_roc.png")

error_base_rate = float(incorrect.mean())
det_df = pd.DataFrame(det_rows); det_df["error_base_rate"] = round(error_base_rate, 4)
det_df.to_csv(TABLE_DIR / "table_sr_error_detection.csv", index=False)

log(det_df)
write_summary("\n[SR] ERROR DETECTION")
write_summary(f"Error base rate (test): {error_base_rate:.4f}")
write_summary(det_df.to_string(index=False))

In [ ]:
# ============================================================
# 29. [SR] Selective prediction — risk–coverage analysis
# ============================================================
corr = np.asarray(correct)
n = len(y_test)

order_conf = np.argsort(-np.asarray(conf))        # most confident first
order_entropy = np.argsort(np.asarray(entropy))   # lowest entropy first
order_oracle = np.argsort(~corr)                  # correct first (best case)
order_random = np.random.RandomState(RANDOM_STATE).permutation(n)

def risk_coverage(order, correct_arr):
    c = correct_arr[order].astype(float)
    cov = np.arange(1, len(c)+1) / len(c)
    sel_acc = np.cumsum(c) / np.arange(1, len(c)+1)
    risk = 1.0 - sel_acc
    return cov, risk, sel_acc, float(np.mean(risk))   # last = AURC

curves = {}
for label, order in [("Confidence", order_conf), ("Entropy", order_entropy),
                     ("Random", order_random), ("Oracle", order_oracle)]:
    curves[label] = risk_coverage(order, corr)

aurc_df = pd.DataFrame([{"ranking": k, "aurc": round(v[3], 4)} for k, v in curves.items()])
aurc_df.to_csv(TABLE_DIR / "table_sr_aurc.csv", index=False)

plt.figure(figsize=(7,5))
for label, (cov, risk, sel_acc, aurc) in curves.items():
    ls = "--" if label in ("Random", "Oracle") else "-"
    plt.plot(cov, risk, ls, label=f"{label} (AURC={aurc:.3f})")
plt.xlabel("Coverage (fraction of cases accepted)")
plt.ylabel("Selective risk (error rate on accepted set)")
plt.title("Risk–Coverage Curve"); plt.legend()
save_fig(FIG_DIR, "fig_sr_risk_coverage.png")

# Selective accuracy vs deferral rate (confidence ranking)
cov, risk, sel_acc, _ = curves["Confidence"]
plt.figure(figsize=(7,5))
plt.plot((1.0 - cov) * 100, sel_acc, linestyle="-")
plt.axhline(accuracy_score(y_test, best_pred), color="gray", linestyle="--",
            label="Full-coverage accuracy")
plt.xlabel("Deferral rate (%)"); plt.ylabel("Selective accuracy (accepted set)")
plt.title("Selective Accuracy vs Deferral Rate (confidence ranking)"); plt.legend()
save_fig(FIG_DIR, "fig_sr_selective_accuracy.png")

# Operating-point table (confidence ranking)
total_errors = int((~corr).sum())
rows = []
for c_level in [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4]:
    k = max(1, int(round(c_level * n)))
    acc_idx, def_idx = order_conf[:k], order_conf[k:]
    sel_bacc = (balanced_accuracy_score(y_test[acc_idx], best_pred[acc_idx])
                if len(np.unique(y_test[acc_idx])) > 1 else float("nan"))
    errs_def = int((~corr[def_idx]).sum()) if len(def_idx) else 0
    rows.append({
        "coverage": c_level,
        "deferral_rate": round(1.0 - c_level, 3),
        "selective_accuracy": round(float(accuracy_score(y_test[acc_idx], best_pred[acc_idx])), 4),
        "selective_balanced_accuracy": round(float(sel_bacc), 4),
        "errors_captured_in_deferred": round(errs_def / max(1, total_errors), 4),
    })
coverage_df = pd.DataFrame(rows)
coverage_df.to_csv(TABLE_DIR / "table_sr_selective_coverage.csv", index=False)

log(aurc_df); log(coverage_df)
write_summary("\n[SR] SELECTIVE PREDICTION")
write_summary(aurc_df.to_string(index=False))
write_summary(coverage_df.to_string(index=False))

In [ ]:
# ============================================================
# 30. [SR] Accept/defer operating points
# ============================================================
corr = np.asarray(correct)
conf_a, ent_a = np.asarray(conf), np.asarray(entropy)

# (a) Paper's fixed rule: accept if conf >= tau_c AND entropy <= tau_H  (Eqs. 16-18)
tau_c, tau_H = 0.70, 0.60
accept_mask = (conf_a >= tau_c) & (ent_a <= tau_H)
def_mask = ~accept_mask
op_rows = [{
    "rule": f"fixed (tau_c={tau_c}, tau_H={tau_H})",
    "coverage": round(float(accept_mask.mean()), 4),
    "accuracy_accepted": (round(float(accuracy_score(y_test[accept_mask], best_pred[accept_mask])), 4)
                          if accept_mask.sum() > 0 else float("nan")),
    "accuracy_deferred": (round(float(accuracy_score(y_test[def_mask], best_pred[def_mask])), 4)
                          if def_mask.sum() > 0 else float("nan")),
}]

# (b) Principled point: largest coverage achieving a target selective accuracy
target_acc = 0.90
_, _, sel_acc_c, _ = curves["Confidence"]
ok = np.where(sel_acc_c >= target_acc)[0]
if len(ok) > 0:
    kbest = int(ok[-1] + 1)
    acc_idx = order_conf[:kbest]; def_idx = order_conf[kbest:]
    op_rows.append({
        "rule": f"target selective acc >= {target_acc} (confidence)",
        "coverage": round(kbest / n, 4),
        "accuracy_accepted": round(float(accuracy_score(y_test[acc_idx], best_pred[acc_idx])), 4),
        "accuracy_deferred": (round(float(accuracy_score(y_test[def_idx], best_pred[def_idx])), 4)
                              if len(def_idx) else float("nan")),
    })
else:
    op_rows.append({"rule": f"target selective acc >= {target_acc}",
                    "coverage": 0.0, "accuracy_accepted": float("nan"),
                    "accuracy_deferred": float("nan")})

op_df = pd.DataFrame(op_rows)
op_df.to_csv(TABLE_DIR / "table_sr_operating_points.csv", index=False)
log(op_df)
write_summary("\n[SR] OPERATING POINTS")
write_summary(op_df.to_string(index=False))

In [ ]:
# ============================================================
# 31. [SR] Explanation stability — add Spearman rank correlation
# ============================================================
from scipy.stats import spearmanr
rho, _ = spearmanr(importance_norm, sensitivity_norm)
stability_ext = pd.DataFrame([{
    "best_model": best_model_name,
    "pearson_importance_sensitivity": round(float(importance_sensitivity_corr), 4),
    "spearman_importance_sensitivity": round(float(rho), 4),
    "importance_sensitivity_mae": round(float(importance_sensitivity_mae), 4),
}])
stability_ext.to_csv(TABLE_DIR / "table_sr_explanation_stability_ext.csv", index=False)
log(stability_ext)
write_summary("\n[SR] EXPLANATION STABILITY (extended)")
write_summary(stability_ext.to_string(index=False))

## Class-Aware Deferral (Scientific Reports revision, second contribution)

The selective-prediction gains in cells 29–30 are **majority-class driven**: global
confidence thresholding keeps easy nevi and defers everything else, so balanced
accuracy on the accepted set does *not* improve. The cells below confront this
directly:

- **Cell 32** quantifies the problem (per-class recall on the accepted set + melanoma miss rate).
- **Cell 33** introduces a **melanoma-sensitive** deferral rule and shows it misses far
  fewer melanomas than global-confidence deferral at the *same* deferral budget.
- **Cell 34** fits **per-class confidence thresholds** and shows they recover balanced
  accuracy that the global threshold cannot.

All cells reuse already-computed probabilities; the only label-dependent assumption is
the (now corrected) class map, with melanoma = label 6.

In [ ]:
# ============================================================
# 32. [SR] Class-aware diagnosis: does global deferral protect minority classes?
# ============================================================
progress("[SR][CLASS] per-class diagnosis of global confidence deferral")

corr = np.asarray(correct)
MEL = [k for k, v in class_names.items() if v == "mel"][0]
n = len(y_test)

diag_rows = []
for c_level in [1.0, 0.8, 0.6, 0.5]:
    k = max(1, int(round(c_level * n)))
    acc_idx = order_conf[:k]
    ya, pa = y_test[acc_idx], best_pred[acc_idx]
    row = {"coverage": c_level}
    for cls in classes:
        m = (ya == cls)
        row[class_names[int(cls)]] = (round(float((pa[m] == cls).mean()), 3)
                                      if m.sum() > 0 else float("nan"))
    # clinically critical: melanomas accepted but called non-melanoma (missed)
    true_mel = (y_test == MEL)
    accepted = np.zeros(n, dtype=bool); accepted[acc_idx] = True
    missed = int(np.sum(true_mel & accepted & (best_pred != MEL)))
    row["mel_missed_frac_of_all_mel"] = round(float(missed / max(1, true_mel.sum())), 3)
    diag_rows.append(row)

diag_df = pd.DataFrame(diag_rows)
diag_df.to_csv(TABLE_DIR / "table_sr_class_aware_diagnosis.csv", index=False)
progress("[SR][CLASS] diagnosis table built")
log(diag_df)
write_summary("\n[SR] CLASS-AWARE DIAGNOSIS (per-class recall on accepted set)")
write_summary(diag_df.to_string(index=False))

In [ ]:
# ============================================================
# 33. [SR] Melanoma-sensitive deferral rule
# ============================================================
progress("[SR][MEL] building melanoma-sensitive deferral curves")

MEL = [k for k, v in class_names.items() if v == "mel"][0]
p_mel = best_proba[:, MEL]
true_mel = (y_test == MEL)
n = len(y_test); n_mel = int(true_mel.sum())

def mel_missed_and_coverage(accept_mask):
    missed = np.sum(true_mel & accept_mask & (best_pred != MEL))
    return float(missed / max(1, n_mel)), float(accept_mask.mean())

# (1) Global confidence-only deferral (accept the most confident fraction)
glob_pts = []
for c_level in np.linspace(0.05, 1.0, 20):
    k = max(1, int(round(c_level * n)))
    am = np.zeros(n, dtype=bool); am[order_conf[:k]] = True
    miss, cov = mel_missed_and_coverage(am)
    glob_pts.append((cov, miss))
glob_pts = np.array(sorted(glob_pts))

# (2) Melanoma-aware rule: DEFER if P(melanoma) >= tau_mel
mel_pts, mel_table = [], []
for tau_mel in [1.01, 0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1, 0.07, 0.05, 0.03]:
    am = ~(p_mel >= tau_mel)
    miss, cov = mel_missed_and_coverage(am)
    mel_pts.append((cov, miss))
    mel_table.append({"tau_mel": tau_mel, "coverage": round(cov, 3),
                      "mel_missed_frac": round(miss, 3),
                      "mel_sensitivity_incl_referral": round(1 - miss, 3)})
mel_pts = np.array(sorted(mel_pts))
mel_df = pd.DataFrame(mel_table)
mel_df.to_csv(TABLE_DIR / "table_sr_melanoma_aware_rule.csv", index=False)

plt.figure(figsize=(7,5))
plt.plot(glob_pts[:,0], glob_pts[:,1], marker="o", label="Global confidence deferral")
plt.plot(mel_pts[:,0], mel_pts[:,1], marker="s", label="Melanoma-sensitive deferral")
plt.xlabel("Coverage (fraction of cases accepted)")
plt.ylabel("Melanoma miss rate (fraction of all true melanomas)")
plt.title("Melanoma Safety vs Coverage: Global vs Class-Aware Deferral")
plt.legend()
save_fig(FIG_DIR, "fig_sr_melanoma_aware_deferral.png")
progress("[SR][MEL] melanoma-aware comparison figure saved")

# Matched-coverage comparison near 0.6 coverage
target_cov = 0.6
j = int(np.argmin(np.abs(mel_pts[:,0] - target_cov)))
cov_m, miss_m = float(mel_pts[j,0]), float(mel_pts[j,1])
k = max(1, int(round(cov_m * n)))
am = np.zeros(n, dtype=bool); am[order_conf[:k]] = True
miss_g, _ = mel_missed_and_coverage(am)
progress(f"[SR][MEL] @coverage~{cov_m:.2f}: class-aware miss={miss_m:.3f} vs global miss={miss_g:.3f}")

log(mel_df)
write_summary("\n[SR] MELANOMA-SENSITIVE DEFERRAL")
write_summary(f"Total true melanomas in test set: {n_mel}")
write_summary(mel_df.to_string(index=False))
write_summary(f"At coverage ~{cov_m:.2f}: melanoma miss rate = {miss_m:.3f} (class-aware) "
              f"vs {miss_g:.3f} (global confidence)")

In [ ]:
# ============================================================
# 34. [SR] Per-class confidence thresholds for balanced reliability
# ============================================================
progress("[SR][PCT] fitting per-class confidence thresholds on calibration split")

conf_all = np.asarray(conf)
pred_all = np.asarray(best_pred)
target_class_acc = 0.85

# Fit one confidence threshold per PREDICTED class on the calibration split (cell 27)
tau_by_class = {}
for cls in classes:
    m = (pred_all[cal_idx] == cls)
    if m.sum() < 10:
        tau_by_class[int(cls)] = 1.01   # too few accepted -> effectively defer this class
        continue
    cvals = conf_all[cal_idx][m]
    cok = (y_test[cal_idx][m] == cls)
    chosen = 1.01
    for t in np.unique(cvals):
        sel = cvals >= t
        if sel.sum() > 0 and cok[sel].mean() >= target_class_acc:
            chosen = float(t); break
    tau_by_class[int(cls)] = chosen

# Apply on the evaluation split
acc_mask = np.array([conf_all[i] >= tau_by_class[int(pred_all[i])] for i in eval_idx])
ya, pa = y_test[eval_idx][acc_mask], pred_all[eval_idx][acc_mask]
cov_pct = float(acc_mask.mean())

# Global threshold matched to the same coverage on the eval split (fair comparison)
conf_eval = conf_all[eval_idx]
kk = max(1, int(round(cov_pct * len(eval_idx))))
order_eval = np.argsort(-conf_eval)
gmask = np.zeros(len(eval_idx), dtype=bool); gmask[order_eval[:kk]] = True
yg, pg = y_test[eval_idx][gmask], pred_all[eval_idx][gmask]

def safe_bacc(yt, yp):
    return float(balanced_accuracy_score(yt, yp)) if len(np.unique(yt)) > 1 else float("nan")

pct_df = pd.DataFrame([
    {"rule": "per-class thresholds", "coverage": round(cov_pct, 3),
     "selective_accuracy": round(float(accuracy_score(ya, pa)), 4),
     "selective_balanced_accuracy": round(safe_bacc(ya, pa), 4)},
    {"rule": "global threshold (matched coverage)", "coverage": round(float(gmask.mean()), 3),
     "selective_accuracy": round(float(accuracy_score(yg, pg)), 4),
     "selective_balanced_accuracy": round(safe_bacc(yg, pg), 4)},
])
pct_df.to_csv(TABLE_DIR / "table_sr_per_class_thresholds.csv", index=False)
progress("[SR][PCT] per-class vs global comparison built")
log({class_names[k]: round(v, 3) for k, v in tau_by_class.items()})
log(pct_df)
write_summary("\n[SR] PER-CLASS THRESHOLDS")
write_summary("Per-class confidence thresholds (by predicted class): "
              + str({class_names[k]: round(v, 3) for k, v in tau_by_class.items()}))
write_summary(pct_df.to_string(index=False))
progress("[SR] all class-aware analyses complete")

In [ ]:
# ============================================================
# 26. Figure inventory and final configuration
# ============================================================

figure_inventory = pd.DataFrame([
    {"figure": "fig_ieee_core_ablation_performance.png", "purpose": "Core ablation comparison"},
    {"figure": "fig_ieee_cv_stability.png", "purpose": "Cross-validation stability"},
    {"figure": "fig_ieee_noise_curve.png", "purpose": "Gaussian-noise robustness"},
    {"figure": "fig_ieee_perturbation_summary.png", "purpose": "Multi-perturbation robustness"},
    {"figure": "fig_ieee_reliability_diagram.png", "purpose": "Calibration and reliability"},
    {"figure": "fig_ieee_calibration_gap.png", "purpose": "Calibration gaps"},
    {"figure": "fig_ieee_confidence_accuracy.png", "purpose": "Confidence-aware decision behavior"},
    {"figure": "fig_ieee_entropy_correctness.png", "purpose": "Uncertainty/correctness relationship"},
    {"figure": "fig_ieee_feature_importance_heatmap.png", "purpose": "Corrected global feature importance"},
    {"figure": "fig_ieee_top_feature_importance_decay.png", "purpose": "Importance concentration"},
    {"figure": "fig_ieee_cumulative_feature_importance.png", "purpose": "Cumulative importance structure"},
    {"figure": "fig_ieee_perturbation_sensitivity_map.png", "purpose": "Perturbation sensitivity"},
    {"figure": "fig_ieee_importance_sensitivity_scatter.png", "purpose": "Explanation-stability relationship"},
    {"figure": "fig_sr_reliability_temperature.png", "purpose": "[SR] Calibration before/after temperature scaling"},
    {"figure": "fig_sr_error_detection_roc.png", "purpose": "[SR] Entropy/confidence misclassification detection"},
    {"figure": "fig_sr_risk_coverage.png", "purpose": "[SR] Risk-coverage / selective prediction"},
    {"figure": "fig_sr_selective_accuracy.png", "purpose": "[SR] Selective accuracy vs deferral rate"},
    {"figure": "fig_sr_melanoma_aware_deferral.png", "purpose": "[SR] Melanoma-sensitive vs global deferral"}
])
figure_inventory.to_csv(TABLE_DIR / "table_ieee_figure_inventory.csv", index=False)

config = {
    "run_id": RUN_ID,
    "data_root": str(DATA_ROOT),
    "csv_path": str(CSV_PATH),
    "output_root": str(OUTPUT_ROOT),
    "best_model": best_model_name,
    "importance_method": importance_method,
    "importance_source_model": importance_source_model,
    "figures_generated": sorted([p.name for p in FIG_DIR.glob("*.png")]),
    "tables_generated": sorted([p.name for p in TABLE_DIR.glob("*.csv")])
}

with open(OTHER_DIR / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

write_summary("\nFIGURE INVENTORY")
write_summary(figure_inventory.to_string(index=False))
write_summary("\nDONE")
write_summary(f"All outputs saved to: {OUTPUT_ROOT}")

log("[DONE] Notebook completed.")
log(f"[DONE] Output folder: {OUTPUT_ROOT}")
log(f"[DONE] Summary: {SUMMARY_PATH}")
